# AiJockey — Env Check Only

Self-contained. No repo upload needed. Validates Colab can run the dep stack.

**Goal**: confirm GPU + Demucs + madmom + CLAP + rubberband all work BEFORE you commit to the full pipeline.

**Time**: ~5 min (mostly checkpoint downloads).

## 1. GPU

Runtime → Change runtime type → **T4 GPU** → Save. Then run:

In [ ]:
!nvidia-smi

## 2. System binaries

In [ ]:
!apt-get install -y rubberband-cli ffmpeg > /dev/null 2>&1
!rubberband --version 2>&1 | head -1
!ffmpeg -version 2>&1 | head -1

## 3. Python deps (Py 3.12 compatible set)

~3-5 min. After this cell completes, **Runtime → Restart runtime** before going to cell 4.

In [ ]:
# numpy<2 + cython<3 first (madmom needs both at build time)
!pip install --force-reinstall --no-deps "numpy<2.0"
!pip install --no-build-isolation --upgrade "cython<3"

# Audio + ML libs
!pip install demucs==4.0.1 librosa==0.10.2 soundfile==0.12.1 \
    pyrubberband pyloudnorm==0.1.1 scipy tqdm transformers

# madmom — Py 3.12 needs git source build
!pip install --no-build-isolation git+https://github.com/CPJKU/madmom.git

# Note: laion-clap intentionally skipped (wheel build fails on Py 3.12).
# CLAP runs via transformers.ClapModel — see src/clap_wrapper.py

print('\n=== INSTALL DONE. Runtime -> Restart runtime, then run remaining cells. ===')

## 4. Import sanity

Run AFTER restarting runtime.

In [ ]:
import importlib
libs = ['torch', 'torchaudio', 'demucs', 'madmom', 'librosa',
        'soundfile', 'pyrubberband', 'pyloudnorm', 'transformers',
        'scipy', 'numpy']
fails = []
for name in libs:
    try:
        m = importlib.import_module(name)
        ver = getattr(m, '__version__', '?')
        print(f'  OK  {name:14s} {ver}')
    except Exception as e:
        fails.append((name, str(e)))
        print(f'FAIL  {name:14s} {e}')

import torch
print(f'\ntorch.cuda.is_available(): {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'device 0: {torch.cuda.get_device_name(0)}')

print('\nAll OK' if not fails else f'{len(fails)} import(s) failed')

## 5. Demucs smoke test

Downloads `htdemucs` checkpoint (~80 MB) on first run.

In [ ]:
import torch, torchaudio

sr = 44100
t = torch.linspace(0, 10, sr * 10)
sig = torch.sin(2 * 3.14159 * 220 * t) * 0.3
wav = torch.stack([sig, sig])

from demucs.pretrained import get_model
from demucs.apply import apply_model
m = get_model('htdemucs').cuda()
m.eval()
with torch.no_grad():
    sources = apply_model(m, wav.unsqueeze(0).cuda(), split=True, overlap=0.25)[0]
print(f'Demucs OK. stems shape: {sources.shape}, sources: {m.sources}')

## 6. madmom smoke test

In [ ]:
import numpy as np
import madmom

sr = 44100
dur = 8.0
n = int(sr * dur)
audio = np.zeros(n, dtype=np.float32)
beat_period = sr // 2  # 120 BPM
for i in range(0, n, beat_period):
    burst = int(0.05 * sr)
    audio[i:i+burst] += np.exp(-np.linspace(0, 5, burst)) * np.sin(
        2 * np.pi * 60 * np.arange(burst) / sr)

proc = madmom.features.beats.RNNBeatProcessor()
act = proc(audio)
tracker = madmom.features.beats.BeatTrackingProcessor(fps=100)
beats = tracker(act)
print(f'madmom OK. detected {len(beats)} beats')
print(f'first 5 beat times: {beats[:5]}')

## 7. CLAP smoke test (via HF transformers)

Downloads CLAP checkpoint (~600 MB) on first run.

In [ ]:
import numpy as np
import torch
from transformers import ClapModel, ClapProcessor

ckpt = 'laion/clap-htsat-unfused'
model = ClapModel.from_pretrained(ckpt)
processor = ClapProcessor.from_pretrained(ckpt)
model.eval()
if torch.cuda.is_available():
    model = model.cuda()

sr = 48000
t = np.linspace(0, 10, sr * 10, dtype=np.float32)
audio = np.sin(2 * np.pi * 440 * t) * 0.3

# transformers 5.x: kwarg `audio`. 4.x: kwarg `audios`. Try both.
try:
    inputs = processor(audio=audio, sampling_rate=sr, return_tensors='pt')
except (TypeError, ValueError):
    inputs = processor(audios=audio, sampling_rate=sr, return_tensors='pt')

if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}
with torch.no_grad():
    out = model.get_audio_features(**inputs)

# transformers 4.x: tensor. 5.x: ModelOutput object — extract embedding.
def extract(out):
    if isinstance(out, torch.Tensor):
        return out
    for attr in ('audio_embeds', 'embeds', 'pooler_output', 'last_hidden_state'):
        v = getattr(out, attr, None)
        if v is not None and hasattr(v, 'shape'):
            if attr == 'last_hidden_state' and v.dim() == 3:
                return v.mean(dim=1)
            return v
    raise RuntimeError(f'cannot extract embedding from {type(out)}; '
                       f'attrs={[a for a in dir(out) if not a.startswith("_")]}')

emb = extract(out)
print(f'CLAP OK. embedding shape: {emb.shape}')

## 8. rubberband + pyrubberband smoke test

In [ ]:
import numpy as np
import pyrubberband as pyrb

sr = 44100
x = np.sin(2 * np.pi * 440 * np.arange(sr * 2) / sr).astype(np.float32)
x = np.stack([x, x], axis=1)

stretched = pyrb.time_stretch(x, sr, 1.5)
shifted = pyrb.pitch_shift(x, sr, 2.0)
print(f'rubberband OK. stretch: {x.shape} -> {stretched.shape}')
print(f'pitch shift OK. shape: {shifted.shape}')

## 9. Verdict

If all 5-8 cells passed → env ready. Move to full pipeline notebook (`aijockey_colab.ipynb`) which needs the actual repo.

If anything failed → fix env before uploading code:

| Failure | Fix |
|---------|-----|
| `madmom` install fails | force `pip install "cython<3" "numpy<2.0"`, restart, then `pip install --no-build-isolation git+https://github.com/CPJKU/madmom.git` |
| `OSError: rubberband` | re-run cell 2 (apt install) |
| Demucs OOM | should not happen on T4 with synthetic 10s — if it does, reset runtime |
| CLAP download timeout | re-run cell 7 (resumes) |
| `numpy>=2.0` errors after install | restart runtime, the install cell forced 1.26 |
| `pyrubberband` build fails on Py 3.12 | use `pip install pyrubberband` (no version pin) — version 0.4.0+ works |